In [1]:
# Paths (relative to the repository root, so the notebooks run on any machine)
# Expected layout:  <repo>/notebooks/ (this file), <repo>/data/, <repo>/models/,
#                   <repo>/figures/, <repo>/results/, <repo>/feature_importance/
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
FI_DIR = ROOT / "feature_importance"
for _d in (DATA_DIR, MODELS_DIR, FIG_DIR, RESULTS_DIR, FI_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

In [3]:
df_all = pd.read_csv(str(DATA_DIR / "Descriptors_Full.csv"),  low_memory=False)

In [4]:
metadata = [
    "row_id",
    "SMILES_Ligands",
    "Counterion",
    "Molecule",
    "Cell_line",
    "Metal",
    "Charge_complex",
    "Oxidation_state",
    "Atomic_number",
    "Valence_e",
    "Time(h)",
    "Year",
    "IC50_Dark_value",
    "pIC50", 
    "Abbreviation_in_the_article", "IC50_Dark(M*10^-6)",
    "IC50_Dark_standard_error(M*10^-6)", "IC50_Light(M*10^-6)",
    "IC50_Light_standard_error(M*10^-6)", "IC50_Light_value",
    "Excitation_Wavelength(nm)", "Irradiation_Time(minutes)",
    "Irradiation_Power(W*m^-2)", "DOI",
    "IC50_Cisplatin(M*10^-6)", "IC50_Cisplatin_standard_error(M*10^-6)",
    "IC50_Cisplatin_value"
]

In [5]:
#Everything else is considered a descriptor.

descriptor_cols = [
    c for c in df_all.columns
    if c not in metadata
]

In [6]:
print(len(descriptor_cols))

1830


In [7]:
#Convert Descriptors to Numeric

#Some Mordred descriptors are stored as strings or custom objects.

descriptor_df = df_all[descriptor_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [8]:
# Split  the dataset.
train_idx, test_idx = train_test_split(
    df_all.index,
    test_size=0.20,
    random_state=42
)

desc_train = descriptor_df.loc[train_idx].copy()
desc_test  = descriptor_df.loc[test_idx].copy()

print(desc_train.shape, desc_test.shape)

(21440, 1830) (5361, 1830)


In [9]:
#Check missing-value filter — threshold computed on TRAIN ONLY
missing_percent = descriptor_df.isnull().mean() * 100

missing_percent.sort_values(ascending=False).head(20)

missing_percent_train = desc_train.isnull().mean() * 100
threshold = 30
keep_cols = missing_percent_train[missing_percent_train < threshold].index

desc_train = desc_train[keep_cols]
desc_test  = desc_test[keep_cols]     # same columns, no re-evaluation on test

print(desc_train.shape, desc_test.shape)

(21440, 1383) (5361, 1383)


In [10]:
# Check for infinities

print("Positive infinity:", np.isposinf(desc_train).sum().sum())
print("Negative infinity:", np.isneginf(desc_train).sum().sum())
print("NaN:", desc_train.isnull().sum().sum())

print("Positive infinity:", np.isposinf(desc_test).sum().sum())
print("Negative infinity:", np.isneginf(desc_test).sum().sum())
print("NaN:", desc_test.isnull().sum().sum())

Positive infinity: 12
Negative infinity: 0
NaN: 178837
Positive infinity: 0
Negative infinity: 0
NaN: 39285


In [11]:
# Replace infinities

desc_train.replace([np.inf, -np.inf], np.nan, inplace=True)
desc_test.replace([np.inf, -np.inf], np.nan, inplace=True)

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb2
12723,5.000068,5.000068,0.000000,0.000000,0.101060,10.195122,606.245,571.973,605.204186,214,...,7.515889,10.211156,93.055453,605.204186,8.069389,53300001217,45,188.0,210.0,9.444444
19067,13.631383,13.631383,0.008944,-4.644445,0.072180,11.441860,1164.331,1107.883,1163.424040,428,...,7.686621,11.424346,143.357468,1163.424040,8.193127,163200023703,137,460.0,544.0,18.791667
13430,5.038360,5.038360,0.000000,0.000000,0.466706,9.769231,392.976,365.760,392.156885,142,...,0.000000,9.610525,59.368614,392.156885,7.399187,17500000568,28,112.0,121.0,5.944444
9163,5.026411,5.026411,0.496104,0.496104,0.063308,10.959184,653.815,622.567,653.236165,236,...,0.000000,10.904138,87.388640,653.236165,8.165452,78400001466,77,260.0,309.0,10.944444
11522,12.758184,12.758184,0.000000,-0.254577,0.158118,17.022222,650.409,583.881,649.471660,260,...,0.000000,10.174316,81.214395,649.471660,5.851096,38400004181,54,200.0,218.0,10.416667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13980,11.717912,11.717912,0.000000,-0.243454,0.407472,10.960000,404.701,386.557,403.039946,138,...,0.000000,9.713960,58.384113,403.039946,9.373022,6900001282,30,106.0,119.0,5.000000
9140,11.358245,11.358245,0.000000,-0.248611,0.456722,9.818182,452.941,431.773,452.130253,164,...,0.000000,10.252770,68.719231,452.130253,8.372782,28400000944,45,162.0,187.0,7.222222
19287,10.056982,10.056982,0.000000,-1.347222,0.489186,10.000000,446.954,421.754,446.153942,166,...,7.167809,10.287286,82.222036,446.153942,7.827262,24100001076,46,158.0,185.0,6.888889
26421,11.242593,11.242593,0.309679,-1.068194,0.196315,12.068182,572.692,542.452,572.245584,214,...,0.000000,10.899495,82.109979,572.245584,7.733048,62000001079,72,238.0,285.0,9.555556


In [12]:
# median imputation — medians computed on TRAIN ONLY
train_medians = desc_train.median()

desc_train = desc_train.fillna(train_medians)
desc_test  = desc_test.fillna(train_medians)   # frozen train medians applied to test

In [13]:
# verify no NaN/inf remain
for name, d in [("train", desc_train), ("test", desc_test)]:
    print(name, "inf+:", np.isposinf(d).sum().sum(),
          "inf-:", np.isneginf(d).sum().sum(),
          "NaN:", d.isnull().sum().sum())

train inf+: 0 inf-: 0 NaN: 0
test inf+: 0 inf-: 0 NaN: 0


In [14]:
# VarianceThreshold — fit on TRAIN ONLY 
selector = VarianceThreshold(threshold=0)
selector.fit(desc_train)                       # fit on train only

selected_columns = desc_train.columns[selector.get_support()]
desc_train = desc_train[selected_columns]
desc_test  = desc_test[selected_columns]       # same columns applied to test

print(desc_train.shape, desc_test.shape)


(21440, 1264) (5361, 1264)


In [15]:
# duplicate-descriptor removal — decided on TRAIN ONLY ---
print("Before:", desc_train.shape)

train_T_dedup = desc_train.T.drop_duplicates()
kept_cols_dedup = train_T_dedup.T.columns      # the surviving column set, decided on train

desc_train = desc_train[kept_cols_dedup]
desc_test  = desc_test[kept_cols_dedup]        # apply the same kept columns to test

print("After:", desc_train.shape)

Before: (21440, 1264)
After: (21440, 1215)


In [16]:
# correlation filtering (r > 0.95) — decided on TRAIN ONLY
corr_matrix = desc_train.corr(method="pearson").abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if (upper[column] > 0.95).any()]
print(f"Columns to remove: {len(to_drop)}")

desc_train = desc_train.drop(columns=to_drop)
desc_test  = desc_test.drop(columns=to_drop)   # same drop list applied to test

print("Remaining descriptors — train:", desc_train.shape, " test:", desc_test.shape)


Columns to remove: 533
Remaining descriptors — train: (21440, 682)  test: (5361, 682)


In [17]:
# save the clean, leakage-free descriptor matrices
desc_train["row_id"] = df_all.loc[train_idx, "row_id"].values
desc_test["row_id"]  = df_all.loc[test_idx, "row_id"].values

desc_train.to_csv(str(DATA_DIR / "Processed_Descriptors_Train.csv"), index=False)
desc_test.to_csv(str(DATA_DIR / "Processed_Descriptors_Test.csv"), index=False)

C:\Users\aadil\AppData\Local\Temp\ipykernel_25252\2960152240.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  desc_train["row_id"] = df_all.loc[train_idx, "row_id"].values
C:\Users\aadil\AppData\Local\Temp\ipykernel_25252\2960152240.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  desc_test["row_id"]  = df_all.loc[test_idx, "row_id"].values


In [18]:
# rebuild metadata + target, per split
pIC50_df = pd.read_csv(str(DATA_DIR / "pIC50.csv"))
full_df  = pd.read_csv(str(DATA_DIR / "Descriptors_Full.csv"), low_memory=False)

meta_cols = [
    "row_id", "SMILES_Ligands", "Counterion", "Cell_line", "Metal",
    "Charge_complex", "Oxidation_state", "Atomic_number", "Valence_e",
    "Time(h)", "Year", "IC50_Dark_value"
]
metadata_df = full_df[meta_cols]

final_train = metadata_df.merge(desc_train, on="row_id").merge(pIC50_df, on="row_id")
final_test  = metadata_df.merge(desc_test,  on="row_id").merge(pIC50_df, on="row_id")

final_train = final_train.drop(columns="row_id")
final_test  = final_test.drop(columns="row_id")

print(final_train.shape, final_test.shape)


(21440, 694) (5361, 694)


In [19]:
# --- Cell 16: sanity check (unchanged from before) ---
for name, d in [("train", final_train), ("test", final_test)]:
    check = -np.log10(d["IC50_Dark_value"] * 1e-6)
    print(name, "max mismatch vs pIC50:", (check - d["pIC50"]).abs().max())

train max mismatch vs pIC50: 1.7763568394002505e-15
test max mismatch vs pIC50: 1.7763568394002505e-15


In [20]:
# --- Cell 17: save final, ready-to-model, leakage-free datasets ---
final_train.to_csv(str(DATA_DIR / "Final_ML_Train.csv"), index=False)
final_test.to_csv(str(DATA_DIR / "Final_ML_Test.csv"), index=False)

print(final_train.shape, final_test.shape)

(21440, 694) (5361, 694)
